# Bonus

🎯 84 feature’dan oluşan tam `ML_Houses_dataset.csv` dataset’iyle [buradan ulaşarak](https://d32aokrjazspmn.cloudfront.net/materials/ML_Houses_dataset.csv) serbestçe çalışabilirsiniz!

- Feature’ları inceleyin
- Uygun şekilde preprocess edin ve encode edin
- Feature engineering için beyin fırtınası yapın
- Bunları modelinize ekleyin
- Feature selection uygulayın

👇 Dosyayı yerel olarak `data` klasörüne kaydedin ve buradan içe aktarın.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Veriyi yükle (aynı klasörde olduğu için doğrudan isimle çağırıyoruz)
df = pd.read_csv("ML_Houses_dataset.csv")

# Eksik değerleri büyükten küçüğe gör
missing = df.isnull().sum().sort_values(ascending=False)
print("Eksik Değer İçeren Sütunlar:")
print(missing[missing > 0].head(20))

# İlk bakış: 84 sütunu ve veri tiplerini gör
df.head()

Eksik Değer İçeren Sütunlar:
WallMat         1755
PoolQC          1751
MiscFeature     1694
Alley           1648
Fence           1418
MasVnrType      1062
FireplaceQu      827
LotFrontage      309
GarageType        93
GarageQual        93
GarageYrBlt       93
GarageFinish      93
GarageCond        93
BsmtFinType2      45
BsmtExposure      44
BsmtQual          43
BsmtFinType1      43
BsmtCond          43
Pesos             13
RoofSurface       10
dtype: int64


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [ ]:
# 1. Kategorik (Metin) Boşlukları Doldurma
# Bu sütunlardaki NaN değerleri aslında "Yok" (None) anlamına geliyor
none_features = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]

for col in none_features:
    df[col] = df[col].fillna("None")

# 2. Sayısal Boşlukları Doldurma
# LotFrontage: Evin caddeye bakan cephesi. Mahallenin medyanını kullanmak mantıklıdır.
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

# GarageYrBlt: Garajı olmayan evlerin garaj yılı 0 olmalı.
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(0)

# RoofSurface ve MasVnrArea gibi sayısal değerler eksikse muhtemelen 0'dır.
numerical_nans = df.select_dtypes(include=['int64', 'float64']).columns
for col in numerical_nans:
    df[col] = df[col].fillna(0)

# 3. Kalan Kategorik Boşluklar (En sık değerle doldur)
cat_nans = df.select_dtypes(include=['object']).columns
for col in cat_nans:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Kalan boş veri sayısı:", df.isnull().sum().sum())

Kalan boş veri sayısı: 0


In [5]:
# Toplam Metrekare (Evin asıl gücü budur)
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

# Evin 'Lüks' Puanı (Havuz, Şömine, Garaj varsa +1 puan)
df['LuxuryScore'] = (df['PoolArea'] > 0).astype(int) + \
                    (df['Fireplaces'] > 0).astype(int) + \
                    (df['GarageCars'] > 0).astype(int)

# Gereksiz Id sütununu atalım, modele gürültü yapmasın
df.drop(columns=['Id'], inplace=True)

In [8]:
# Sadece sayısal sütunları baz alarak korelasyon hesapla
correlation_matrix = df.corr(numeric_only=True)
print(correlation_matrix['SalePrice'].sort_values(ascending=False).head(10))

SalePrice      1.000000
Pesos          0.950671
OverallQual    0.792108
TotalSF        0.762620
GrLivArea      0.699717
GarageCars     0.638577
GarageArea     0.619462
TotalBsmtSF    0.604485
1stFlrSF       0.603921
FullBath       0.558821
Name: SalePrice, dtype: float64


In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# 1. Pesos ve SalePrice'ı X'ten çıkarıyoruz
# Ayrıca korelasyonu düşük olan 70'den fazla sütunu eleyip sadece en güçlüleri alalım
features = ['OverallQual', 'TotalSF', 'GrLivArea', 'GarageCars', 'FullBath', 'YearBuilt']
X = df[features]
y = df['SalePrice']

# 2. Modeli Test Et (Cross Validation)
model = LinearRegression()
scores = cross_val_score(model, X, y, cv=5)

print(f"Modelin Ortalama Başarısı (R2): {scores.mean():.4f}")
print(f"Kullanılan Özellikler: {features}")

# 3. Küçük bir tahmin denemesi
model.fit(X, y)
sample_house = np.array([[8, 2500, 1800, 2, 2, 2010]]) # Kalite 8, 2500m2 toplam alan vb.
prediction = model.predict(sample_house)
print(f"\nÖrnek Ev Tahmini Fiyatı: ${prediction[0]:,.2f}")

Modelin Ortalama Başarısı (R2): 0.7472
Kullanılan Özellikler: ['OverallQual', 'TotalSF', 'GrLivArea', 'GarageCars', 'FullBath', 'YearBuilt']

Örnek Ev Tahmini Fiyatı: $242,613.34


/home/cem/.pyenv/versions/3.12.9/envs/workintech/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [ ]:
# 1. En güçlü sayısal özellikler + Kritik kategorik özellikler
important_features = [
    'OverallQual', 'TotalSF', 'GrLivArea', 'GarageCars',
    'FullBath', 'YearBuilt', 'Neighborhood', 'KitchenQual'
]

X_advanced = df[important_features].copy()

# 2. Kategorik özellikleri sayıya çevir (One-Hot Encoding)
X_advanced = pd.get_dummies(X_advanced, columns=['Neighborhood', 'KitchenQual'], drop_first=True)

y = df['SalePrice']

# 3. Yeni Model Skoru
model_advanced = LinearRegression()
advanced_scores = cross_val_score(model_advanced, X_advanced, y, cv=5)

print(f"Gelişmiş Model Skoru (R2): {advanced_scores.mean():.4f}")

# 4. Hatasız Tahmin (Sütun isimlerini koruyarak)
model_advanced.fit(X_advanced, y)
# Test için ilk satırı kullanalım (Hata vermez)
sample_house_valid = X_advanced.iloc[[0]]
prediction = model_advanced.predict(sample_house_valid)
print(f"Gerçekçi Tahmin: ${prediction[0]:,.2f}")

Gelişmiş Model Skoru (R2): 0.8025
Gerçekçi Tahmin: $207,760.30


ℹ️ Dataset’in açıklamasına mutlaka [buradan](https://drive.google.com/file/d/1qLxeQXufW_-KHOckpUweLPhitzjnP7H3/view?usp=sharing) referans verin.